[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LSDtopotools/lsdtt3_notebooks/blob/main/hydrology/s_hype_simulation.ipynb)

# A simple S-HYPE simulation: Ettrick Water, Southern Uplands of Scotland

This notebook runs **S-HYPE**, a daily rainfall–runoff model, on one small catchment and compares the result with the measured river flow. It is the simplest possible set-up: one catchment, the same soil and land use everywhere, and parameters that have not been calibrated. Its purpose is to show how the pieces fit together.

The catchment is the **Ettrick Water above Brockhoperig** (gauging station 21017 of the UK [National River Flow Archive](https://nrfa.ceh.ac.uk/data/station/info/21017), NRFA), 37.5 km² of the Southern Uplands, 258–691 m above sea level.

The workflow is:
1. Install `lsdviztools3` (plotting, from PyPI) and the `lsdtt3` command-line tools (from the latest GitHub release).
2. Get the gauge from the NRFA: its location, catchment area and daily flows.
3. Download a **30 m** DEM with `lsdtt-fetch-raster`.
4. Map the **catchment**: its outline, its channel network and its hillshade, and where it is.
5. Download daily **rainfall and temperature** for the catchment from Open-Meteo.
6. Make the land-use and soil maps and the parameter tables: **uniform** across the catchment.
7. Run S-HYPE with `lsdtt-distributed-hydrology` (`column_model: hype`).
8. Compare the simulated flow with the NRFA record.
9. **Calibrate** seven parameters against the gauge with SPOTPY: a quick run of a few minutes, checked on a year it did not see.
10. Add a **second soil class** along the channel network, and see what a spatial element does.

Each `lsdtt3` program is controlled by a small **parameter file**: a text file of `key: value` lines. We write these files from Python and then run the programs with `!`.

The next section explains what S-HYPE is, how it divides a catchment, and what it needs; skip to **Setup** if you want to run it first.

## How S-HYPE works, and what it needs

### HYPE and S-HYPE

**HYPE** (Hydrological Predictions for the Environment) is the hydrological and water-quality model of the Swedish Meteorological and Hydrological Institute, SMHI (Lindström et al., 2010). **S-HYPE** is its national set-up for Sweden: about 450,000 km² divided into sub-basins averaging 28 km², run every day to give discharge in gauged and ungauged rivers alike (Strömqvist et al., 2012).

`lsdtt3` reimplements HYPE's land and river processes with HYPE's **default model options**, from the HYPE 5.36.0 source code, and tests them against output from HYPE itself. It leaves out lakes, regional groundwater, frozen soil and the water-quality parts.

Every day, for each class of each sub-basin, the model:

* splits precipitation into rain and snow by air temperature, and melts snow with a degree-day factor;
* loses a fixed fraction of precipitation to interception, then fills up to three soil layers;
* drains water above field capacity down to the next layer (percolation) and sideways to the stream (soil runoff, faster from the top layer and on steeper ground);
* removes evapotranspiration, estimated from air temperature with a seasonal factor;
* routes the runoff through the sub-basin's local river and main river: a delay, then a damping store.

### Sub-basins and classes

HYPE is **semi-distributed**. Its spatial unit is the **sub-basin**, and every sub-basin is described by a small set of numbers (HYPE's `GeoData`): its area, its mean elevation and slope, the lengths of its local and main rivers, the fraction of its area in each class, and which sub-basin it drains into.

A **class** is a pair of a land use and a soil type (HYPE's `GeoClass`). Within a sub-basin, HYPE knows **how much** of the area each class covers, but not **where** it is. Each class runs its own soil column, its runoff is weighted by its area fraction, and the sum enters the sub-basin's rivers. Water then flows from sub-basin to sub-basin, upstream first, until it reaches the outlet.

In S-HYPE the sub-basins come from national catchment and river data. In `lsdtt3` they come from the DEM.

### How lsdtt3 builds the sub-basins from the DEM

`lsdtt-distributed-hydrology` works out the whole drainage structure itself, using the same tools as the rest of `lsdtt3`:

1. **Fill** the pits in the DEM, so water can always flow downhill to the edge.
2. **Route flow** with the D8 method: every pixel drains to its steepest neighbour. This gives each pixel its drainage area and its flow distance to the outlet.
3. **Extract channels**: a channel starts where `threshold_contributing_pixels` pixels drain to a point.
4. **Place the outlets.** Here, the NRFA gauge, snapped onto the nearest channel pixel. You can add more points (tributary gauges, confluences), or take every junction of one stream order instead.
5. **Divide the land into incremental sub-basins**: each pixel belongs to the first outlet downstream of it, and each sub-basin drains into the next outlet down (HYPE's `maindown`). Land that reaches the edge of the DEM without passing an outlet is outside the model.
6. **Derive each sub-basin's `GeoData`** from the DEM:
    * the area, and the mean and spread of elevation;
    * each class's area fraction, and its mean elevation relative to the sub-basin's. That sets its temperature, through the lapse rate `tcalt`;
    * the mean slope, as a percentage, 100 × the mean gradient. In HYPE's default model this is the only direct topographic control on runoff: soils drain faster on steeper ground, through `rrcs3`;
    * the main river length: the flow distance along the sub-basin's trunk channel;
    * the local river length: the mean flow distance from the sub-basin's pixels to that trunk.

With `write_hype_geodata: true` this set-up is also written as HYPE's own `GeoData.txt` and `GeoClass.txt`.

### What the model needs

| Input | Parameter-file key | In this notebook |
|---|---|---|
| A DEM | `read_fname` | Copernicus GLO-30, 30 m |
| Daily precipitation (mm), one series for the whole domain | `rainfall_fname` | Open-Meteo (reanalysis), at the centre of the catchment |
| Daily air temperature (°C), same length | `temperature_fname` | Open-Meteo |
| The date of the first row | `hype_start_date` | 2021-10-01 |
| Potential evaporation (optional) | `pet_fname` | not given: HYPE's temperature-based estimate is used |
| A land-use raster (integer codes) | `landcover_fname` | code 1 everywhere |
| A soil raster (integer codes) | `hype_soil_class_fname` | code 1 everywhere |
| General, land-use, soil and class parameter tables | `hype_*_fname` | Step 5 |
| The sub-basin outlets | `basin_outlet_fname` | the NRFA gauge |

### Does land use matter, and how does it become a raster?

Yes. In HYPE's default model, land use sets:

* how much precipitation is lost to **interception** (`pcluse`);
* when snow **melts**, and how fast (`ttmp`, `cmlt`);
* **potential evaporation** (`cevp`), which controls how much water is lost to the atmosphere in summer;
* how quickly water ponding on saturated ground runs off (`srrcs`).

Forest and grassland, for example, differ mainly in interception and evaporation, so a catchment's land use changes both how much water reaches the river and when.

To use it, the land cover has to become an **integer raster on exactly the DEM's grid**:

1. Start from a land-cover map: for the UK, the UKCEH Land Cover Map; for Europe, CORINE.
2. Reproject and resample it onto the DEM grid. A categorical map needs nearest-neighbour resampling, never averaging.
3. Reclassify its many classes into the few that the parameters can tell apart, each with an integer code. For example, 1 = grassland and moorland, 2 = forest, 3 = arable, 4 = urban.
4. Give each code a row in the land-use table, and each (land use, soil) pair that occurs a row in the class table. `lsdtt3` stops with a list of any missing pairs.

Soils go the same way, from a soil map into `hype_soil_class_fname` and the soil table. The [lsdtt-distributed-hydrology reference](https://lsdtopotools.github.io/lsdtt3/cli_reference/cli_reference_lsdtt_distributed_hydrology.html#soil-data) lists UK soil sources and starting values for each soil texture.

`lsdtt3` then counts, for every sub-basin, the pixels of each class: these are the class fractions in `GeoData`.

**This notebook keeps it as simple as possible:** one land use and one soil, so one class covering the whole catchment, and the model is effectively **lumped**. The NRFA's land-cover statistics say the Ettrick catchment is about half grassland, with much of the rest moorland and forestry. Mapping those would be the natural next step.

The equations are in the [lsdtt3 documentation](https://lsdtopotools.github.io/lsdtt3/design_and_algorithms/hydrological_modelling.html), and every input is described in the [lsdtt-distributed-hydrology reference](https://lsdtopotools.github.io/lsdtt3/cli_reference/cli_reference_lsdtt_distributed_hydrology.html).

## Setup

### First set up condacolab

**`condacolab.install()` restarts the Colab runtime.** You will see a message that the session crashed: this is expected. Wait for it to reconnect, then carry on with the next cell.

In [ ]:
!pip install -q condacolab

In [ ]:
import condacolab
condacolab.install()

### Install PyGMT

`lsdviztools3` draws its maps with PyGMT, which needs the GMT library. pip cannot install GMT, so we install PyGMT (and GMT with it) from conda-forge. This is the slowest step: repeated tests suggest it takes around 70 seconds.

In [ ]:
# mamba prints hundreds of lines, so we hide them and only show the end of
# the output if something goes wrong.
import subprocess

result = subprocess.run(["mamba", "install", "-y", "pygmt"], capture_output=True, text=True)
if result.returncode == 0:
    print("pygmt installed successfully.")
else:
    print("pygmt installation FAILED. The last lines of the output were:\n")
    print("\n".join((result.stdout + result.stderr).splitlines()[-30:]))

### Install lsdviztools3 from PyPI

`lsdviztools3`, the plotting package, is on [PyPI](https://pypi.org/project/lsdviztools3/). It needs Python 3.12 or later, which Colab has.

In [ ]:
!pip install -q "lsdviztools3>=0.4.2"

### Get the lsdtt3 command-line tools

The Linux package of the latest [lsdtt3 release](https://github.com/LSDtopotools/lsdtt3/releases/latest). `releases/latest/download/` always points at the newest version.

In [ ]:
!wget -q https://github.com/LSDtopotools/lsdtt3/releases/latest/download/lsdtt3-backend-linux-x86_64-core.tar.gz
!tar -xzf lsdtt3-backend-linux-x86_64-core.tar.gz

Now we tell the system where to find the `lsdtt3` programs and the libraries they need. If your runtime restarts later, run this cell again.

In [ ]:
import glob
import os

root = sorted(glob.glob("/content/lsdtt3-backend-linux-x86_64-core-v*"))[-1]
print("Using", root)
os.environ["PATH"] = f"{root}/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = f"{root}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

# Do NOT set GDAL_DATA / PROJ_DATA / PROJ_LIB here. The lsdtt3 programs find
# their own bundled PROJ and GDAL data, and pointing the Python session at the
# backend's proj.db breaks rasterio and pyproj, which lsdviztools3 uses.

In [ ]:
!lsdtt-distributed-hydrology -v

## Step 1: The gauge, from the National River Flow Archive

The NRFA publishes its station information and daily flows through a web service that needs no account or key. We ask it for station 21017: its location (the catchment outlet), its catchment area, and the gauged daily flow (in m³ s⁻¹) for the period we will simulate.

We simulate three **water years** (1 October to 30 September), 2021–22 to 2023–24. The first year is the **spin-up**: the model starts with its soils at field capacity and its rivers empty, and needs time to settle. We only compare the last two years with the observations.

In [ ]:
import json
import urllib.request

import numpy as np
import pandas as pd

station = 21017
start_date, end_date = "2021-10-01", "2024-09-30"
evaluation_start = "2022-10-01"     # after one year of spin-up

NRFA = "https://nrfaapps.ceh.ac.uk/nrfa/ws"
info = json.load(urllib.request.urlopen(
    f"{NRFA}/station-info?station={station}&format=json-object"
    "&fields=id,name,catchment-area,latitude,longitude,minimum-altitude,maximum-altitude,saar-1961-1990"))["data"][0]
for key, value in info.items():
    print(f"{key:20s} {value}")

# The gauged daily flow: alternating dates and values
series = json.load(urllib.request.urlopen(
    f"{NRFA}/time-series?station={station}&data-type=gdf&format=json-object"
    f"&start-date={start_date}&end-date={end_date}"))["data-stream"]
observed = pd.DataFrame({"date": pd.to_datetime(series[0::2]), "flow_m3s": series[1::2]})
print(f"\n{len(observed)} days of gauged flow, mean {observed.flow_m3s.mean():.2f} m3/s")

# The gauge as a basin outlet for lsdtt3: latitude and longitude in a CSV file
pd.DataFrame({"latitude": [info["latitude"]], "longitude": [info["longitude"]],
              "point_id": [station]}).to_csv("ettrick_gauge.csv", index=False)

## Step 2: Download a 30 m DEM

`lsdtt-fetch-raster` downloads the Copernicus GLO-30 DEM (`dem_source: cop30_aws`, about 30 m, no account needed) for a box that contains the whole catchment above the gauge, with a margin of about 2 km, and projects it to the UTM zone of the box (zone 30N here), so distances are in metres. We also make a hillshade for the map at the end.

In [ ]:
# Lower left (south-west) and upper right (north-east) corners, as copied from Google Maps
lower_left = (55.32, -3.36)
upper_right = (55.45, -3.15)

with open("ettrick_fetch.param", "w") as f:
    f.write("write_fname: ettrick\n")
    f.write(f"west: {lower_left[1]}\n")
    f.write(f"east: {upper_right[1]}\n")
    f.write(f"south: {lower_left[0]}\n")
    f.write(f"north: {upper_right[0]}\n")
    f.write("dem_source: cop30_aws\n")

!lsdtt-fetch-raster ./ ettrick_fetch.param

with open("ettrick_hillshade.param", "w") as f:
    f.write("read_fname: ettrick_DEM.tif\n")
    f.write("write_fname: ettrick_hs\n")
    f.write("write_output_format: gtiff\n")
    f.write("write_hillshade_raster: true\n")

!lsdtt-raster-preprocessing ./ ettrick_hillshade.param

## Step 3: The catchment and its channels

Before modelling, look at the catchment. Two `lsdtt3` tools, run on the DEM:

* `lsdtt-basin-extraction` snaps the gauge onto the channel network and writes the catchment upstream of it as a polygon (`select_basins_from_outlets`, `write_basin_outlines_as_polygon`).
* `lsdtt-channel-extraction` writes the channel network as lines, with each link's Strahler stream order (`write_channel_network_lines`).

Both start channels where `threshold_contributing_pixels` = 500 pixels (0.45 km²) drain to a point, the same as the model will in Step 6, so the map shows the drainage structure the model uses.

In [ ]:
with open("ettrick_basin.param", "w") as f:
    f.write("read_fname: ettrick_DEM.tif\n")
    f.write("write_fname: ettrick_basin\n")
    f.write("threshold_contributing_pixels: 500\n")
    f.write("select_basins_from_outlets: true\n")
    f.write("basin_outlet_fname: ettrick_gauge.csv\n")
    f.write("basin_outlet_snapping_method: expanding_square\n")
    f.write("basin_outlet_contributing_pixels_threshold: 500\n")
    f.write("write_basin_outlines: true\n")
    f.write("write_basin_outlines_as_polygon: true\n")
    f.write("basin_outlines_write_vector_format: flatgeobuf\n")

!lsdtt-basin-extraction ./ ettrick_basin.param

with open("ettrick_channels.param", "w") as f:
    f.write("read_fname: ettrick_DEM.tif\n")
    f.write("write_fname: ettrick_channels\n")
    f.write("threshold_contributing_pixels: 500\n")
    f.write("write_channel_network_lines: true\n")

!lsdtt-channel-extraction ./ ettrick_channels.param

The map: the hillshade, the catchment (orange), the channels inside it (wider for higher stream orders), the gauge (red triangle), and a globe showing where it is. We draw it with `lsdviztools3`'s Python functions, adding one layer at a time to the same figure.

In [ ]:
import geopandas as gpd
from IPython.display import Image, display

from lsdviztools3.io.raster import get_crs, load_raster, to_coordinates, to_gmt_grid
from lsdviztools3.render.base import grid_to_display, save_figure
from lsdviztools3.render.basemap import add_locator_inset
from lsdviztools3.render.basins import render_basins
from lsdviztools3.render.channels import render_channels
from lsdviztools3.render.decorations import add_scalebar
from lsdviztools3.render.hillshade import render_hillshade
from lsdviztools3.render.points import render_points
from lsdviztools3.render.style import MapStyle

catchment = gpd.read_file("ettrick_basin_junction_basin_polygons.fgb")
network = gpd.read_file("ettrick_channels_channel_network_lines.fgb")
channels = gpd.clip(network, catchment)          # only the channels inside the catchment
gauge_points = gpd.GeoDataFrame(pd.read_csv("ettrick_gauge.csv"),
                                geometry=gpd.points_from_xy([info["longitude"]], [info["latitude"]]),
                                crs="EPSG:4326")
print(f"Catchment from the DEM: {catchment.area.sum() / 1e6:.1f} km2 (NRFA: {info['catchment-area']} km2)")
print(f"Channel links in the catchment: {len(channels)}, highest stream order {channels.stream_order.max()}")

style = MapStyle(cmap="gray", transparent=False)
fig = render_hillshade("ettrick_DEM.tif", coordinates="local", shade=True, style=style)
fig = render_basins(catchment, coordinates="local", fill="orange", transparency=75, fig=fig,
                    style=MapStyle(line_color="black", line_width="1p"))
for order, width in [(1, "0.5p"), (2, "0.9p"), (3, "1.4p"), (4, "2.0p"), (5, "2.6p")]:
    links = channels[channels.stream_order == order]
    if not links.empty:
        fig = render_channels(links, coordinates="local", fig=fig,
                              style=MapStyle(line_color="0/90/200", line_width=width))
fig = render_points(gauge_points, symbol="t", pen="0.8p,black", coordinates="local", fig=fig,
                    style=MapStyle(line_color="red", point_size="0.45c"))

dem_local = to_coordinates(load_raster("ettrick_DEM.tif"), "local")
_, extent = grid_to_display(to_gmt_grid(dem_local), get_crs(dem_local))
add_scalebar(fig, extent, get_crs(dem_local), anchor="BR")
add_locator_inset(fig, "ettrick_DEM.tif", mode="global", position="TL", width="3.5c")

save_figure(fig, "ettrick_catchment_map.png", style=style)
display(Image("ettrick_catchment_map.png", width=700))

And a regional map: the borders, rivers and relief around the DEM (red box), from `lsdviz3-basemap`.

In [ ]:
!lsdviz3-basemap --in ettrick_DEM.tif --out ettrick_locator.png --mode regional --relief --figure-size 10c

display(Image("ettrick_locator.png", width=500))

**What the catchment is like.** A steep upland valley in the Southern Uplands, about 13 km long: the Ettrick Water runs north-east along it to the gauge, fed by short, steep side streams from the hills on either side. The rocks are greywackes and shales, under thin, often peaty soils, mostly rough grassland and moorland, with some forestry (the NRFA's land-cover statistics put grassland at about half). Rain falls on steep slopes over thin soils, so the river responds within hours to a day: daily steps catch the response, but not the shape of a flood peak.

The catchment from the DEM is about 4% larger than the NRFA's 37.5 km². A small side stream joins within a pixel of the gauge, so at 30 m the gauge's own pixel already includes it. The difference is within what a 30 m DEM and the gauge's grid reference can resolve.

## Step 4: Rainfall and temperature

S-HYPE is driven by a **daily time series** of precipitation (mm) and air temperature (°C). One series drives the whole catchment.

We get them from the [Open-Meteo historical weather API](https://open-meteo.com/en/docs/historical-weather-api), which needs no key. It serves reanalysis data (ERA5 and higher-resolution models) for any point; we ask for the middle of the catchment. Two checks on what we get:

* **Rainfall.** The NRFA gives the catchment's long-term average annual rainfall (`saar-1961-1990` above, 1740 mm). Compare it with the total below.
* **A caution.** A reanalysis smooths the hills. Upland rainfall is often heavier than any coarse grid suggests, and the gauged flow (Step 1) is a check: if more water leaves the catchment than falls as rain, the rainfall is too low.

The model reads two CSV files: `rainfall_mm` in one, `temperature_c` in the other, one row per day, starting on `hype_start_date`.

In [ ]:
centre_lat, centre_lon = 55.38, -3.27     # the middle of the catchment

url = ("https://archive-api.open-meteo.com/v1/archive"
       f"?latitude={centre_lat}&longitude={centre_lon}&start_date={start_date}&end_date={end_date}"
       "&daily=precipitation_sum,temperature_2m_mean&timezone=GMT")
daily = json.load(urllib.request.urlopen(url))["daily"]
met = pd.DataFrame({"date": pd.to_datetime(daily["time"]),
                    "rainfall_mm": daily["precipitation_sum"],
                    "temperature_c": daily["temperature_2m_mean"]})
assert met.notna().all().all(), "The weather series has gaps"

met[["date", "rainfall_mm"]].to_csv("ettrick_rainfall.csv", index=False)
met[["date", "temperature_c"]].to_csv("ettrick_temperature.csv", index=False)

n_years = len(met) / 365.25
area_m2 = info["catchment-area"] * 1e6
runoff_mm = observed.flow_m3s.sum() * 86400 / area_m2 * 1000
print(f"{len(met)} days from {met.date.min().date()} to {met.date.max().date()}")
print(f"Rainfall:        {met.rainfall_mm.sum() / n_years:6.0f} mm per year")
print(f"Gauged runoff:   {runoff_mm / n_years:6.0f} mm per year")
print(f"Mean temperature: {met.temperature_c.mean():5.1f} C")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
ax1.bar(met.date, met.rainfall_mm, width=1.0, color="tab:blue")
ax1.set_ylabel("Rainfall (mm/day)")
ax1.set_title("Daily forcing for the catchment")
ax2.plot(met.date, met.temperature_c, color="tab:red", lw=0.8)
ax2.axhline(0, color="grey", lw=0.5)
ax2.set_ylabel("Temperature (°C)")
plt.tight_layout()
plt.show()

## Step 5: Land use, soil and parameters: uniform

As described above, S-HYPE divides each sub-basin into **classes**, pairs of a land use and a soil type, each with its own parameters. A full set-up would take land use from a land-cover map and soils from a soil map. To keep this example simple we use **one land use and one soil everywhere**: two maps on the DEM grid with the value 1 in every cell, and a single class.

The maps must be integer rasters on exactly the DEM's grid. We copy the DEM's grid with `rasterio`.

In [ ]:
import rasterio

with rasterio.open("ettrick_DEM.tif") as src:
    dem = src.read(1)
    profile = src.profile
    valid = dem != src.nodata

profile.update(dtype="int32", nodata=-9999, count=1)
for name in ("ettrick_landuse.tif", "ettrick_soil.tif"):
    with rasterio.open(name, "w", **profile) as dst:
        dst.write(np.where(valid, 1, -9999).astype("int32"), 1)
print("Wrote ettrick_landuse.tif and ettrick_soil.tif: code 1 everywhere")

### The parameter tables

Four CSV tables, with HYPE's own parameter names. The values below are not calibrated for this catchment: they are a plausible starting point (the set used in `lsdtt3`'s own tests).

**General parameters** (`name,value`):

| Name | Value | Meaning |
|---|---|---|
| `ttpd`, `ttpi` | 0.5, 1.5 °C | rain/snow threshold offset, and the half-width of the mixed rain-and-snow range |
| `tcalt` | 0.6 °C per 100 m | temperature lapse rate |
| `lp` | 0.75 | below this fraction of field capacity, evapotranspiration is reduced |
| `epotdist` | 4 m⁻¹ | how fast evapotranspiration decreases with depth |
| `whcsnow` | 0.08 | liquid water the snowpack can hold |
| `rrcs3` | 0.002 | how much faster soils drain on steeper slopes (per % slope) |
| `rivvel`, `damp` | 0.7 m s⁻¹, 0.3 | river velocity, and the share of the travel time spent in the damping store |
| `cevpam`, `cevpph` | 0.3, 20 days | seasonal swing of potential evaporation |

**Land use** (one row, code 1): melt and evaporation start above `ttmp` = 0 °C; snow melts at `cmlt` = 3.5 mm °C⁻¹ day⁻¹; potential evaporation is `cevp` = 0.2 mm °C⁻¹ day⁻¹; `srrcs` = 0.15 of water ponding on a saturated surface runs off each day; `pcluse` = 5% of precipitation is lost to interception.

**Soil** (one row, code 1): the water in each layer as volume fractions, **stacked**: 8% held below wilting point (`wcwp`), 22% between wilting point and field capacity (`wcfc`, what plants can use), and 14% above field capacity that can drain (`wcep`). `rrcs1` = 0.25 and `rrcs2` = 0.02 per day are how fast the top and bottom layers drain to the stream; `mperc1`, `mperc2` cap percolation between layers; `macrate`, `srrate`, `mactrinf`, `mactrsm` send part of heavy inputs through macropores and over the surface.

**Class** (one row): land use 1 on soil 1, three soil layers whose bottoms are at 0.1, 0.5 and 1.5 m, and a stream depth of 1 m (soils drain to the stream only above this depth).

In [ ]:
pd.DataFrame({
    "name":  ["ttpd", "ttpi", "tcalt", "lp", "epotdist", "whcsnow", "rrcs3", "rivvel", "damp", "cevpam", "cevpph"],
    "value": [0.5,    1.5,    0.6,     0.75, 4.0,        0.08,      0.002,   0.7,      0.3,    0.3,      20],
}).to_csv("hype_general.csv", index=False)

pd.DataFrame([{"code": 1, "ttmp": 0.0, "cmlt": 3.5, "cevp": 0.2, "srrcs": 0.15, "pcluse": 0.05}]
             ).to_csv("hype_landuse.csv", index=False)

pd.DataFrame([{"code": 1, "wcwp": 0.08, "wcfc": 0.22, "wcep": 0.14,
               "rrcs1": 0.25, "rrcs2": 0.02, "mperc1": 20, "mperc2": 8,
               "macrate": 0.15, "srrate": 0.1, "mactrinf": 12, "mactrsm": 0.6}]
             ).to_csv("hype_soil.csv", index=False)

pd.DataFrame([{"class_id": 1, "landuse": 1, "soil": 1, "n_layers": 3,
               "layer1_bottom_m": 0.1, "layer2_bottom_m": 0.5, "layer3_bottom_m": 1.5,
               "stream_depth_m": 1.0}]
             ).to_csv("hype_classes.csv", index=False)

for name in ("hype_general.csv", "hype_landuse.csv", "hype_soil.csv", "hype_classes.csv"):
    print(f"--- {name}")
    print(open(name).read())

## Step 6: Run S-HYPE

`lsdtt-distributed-hydrology` with `column_model: hype`:

1. routes flow over the filled DEM and builds a channel network (channels start where `threshold_contributing_pixels` = 500 pixels, 0.45 km², drain to a point);
2. snaps the gauge onto the nearest channel (`basin_outlet_snapping_method: expanding_square`) and takes **everything that drains to it** as the model domain: one sub-basin here. The rest of the DEM is outside the model;
3. works out the sub-basin's properties from the DEM: area, mean elevation, mean slope, river lengths;
4. runs the soil column for every day of the forcing, and routes the runoff to the outlet.

In [ ]:
with open("ettrick_hype.param", "w") as f:
    f.write("read_fname: ettrick_DEM.tif\n")
    f.write("write_fname: ettrick\n")
    f.write("write_output_format: gtiff\n")

    # The model
    f.write("column_model: hype\n")
    f.write(f"hype_start_date: {start_date}\n")

    # Forcing
    f.write("rainfall_fname: ettrick_rainfall.csv\n")
    f.write("temperature_fname: ettrick_temperature.csv\n")

    # Land use, soil and parameters
    f.write("landcover_fname: ettrick_landuse.tif\n")
    f.write("hype_soil_class_fname: ettrick_soil.tif\n")
    f.write("hype_general_parameters_fname: hype_general.csv\n")
    f.write("hype_landuse_parameters_fname: hype_landuse.csv\n")
    f.write("hype_soil_parameters_fname: hype_soil.csv\n")
    f.write("hype_classes_fname: hype_classes.csv\n")

    # Channels and the sub-basin outlet (the gauge)
    f.write("threshold_contributing_pixels: 500\n")
    f.write("select_basins_from_outlets: true\n")
    f.write("basin_outlet_fname: ettrick_gauge.csv\n")
    f.write("basin_outlet_snapping_method: expanding_square\n")
    f.write("basin_outlet_contributing_pixels_threshold: 500\n")

    # Extra outputs
    f.write("write_hype_subbasin_raster: true\n")

!lsdtt-distributed-hydrology ./ ettrick_hype.param

## Step 7: The results

The run writes:

| File | Content |
|---|---|
| `ettrick_hype_discharge.csv` | one row per day: the sub-basin's outflow `cout_1` (m³ s⁻¹) and runoff `crun_1` (mm) |
| `ettrick_hype_subbasins.csv` | the sub-basin's properties, worked out from the DEM |
| `ettrick_hype_water_balance.csv` | the catchment's water balance over the run (m³) |
| `ettrick_hype_subbasins.tif` | the sub-basin on the DEM grid |

First, the sub-basin the model built: the same catchment as the map in Step 3.

In [ ]:
subbasins = pd.read_csv("ettrick_hype_subbasins.csv")
print(subbasins.T)
print(f"\nModel catchment area: {subbasins.area_m2.sum() / 1e6:.1f} km2 (NRFA: {info['catchment-area']} km2)")

The water balance: over the three years, the rain that fell is the water that evaporated, plus the water that left through the river, plus the change in storage (soil, snow and rivers). The closure error should be tiny.

In [ ]:
balance = pd.read_csv("ettrick_hype_water_balance.csv").set_index("quantity")["value"]
area = subbasins.area_m2.sum()
print((balance / area * 1000 / n_years).round(1).rename("mm per year"))

### Simulated against observed flow

We compare the two only after the spin-up year, with two common measures of fit (both are 1 for a perfect simulation):

* **NSE** (Nash–Sutcliffe efficiency): 1 minus the squared errors divided by the variance of the observations. 0 means no better than the mean flow.
* **KGE** (Kling–Gupta efficiency): combines the correlation `r`, the ratio of the means (bias) and the ratio of the spreads.

In [ ]:
simulated = pd.read_csv("ettrick_hype_discharge.csv", parse_dates=["date"])
flows = observed.merge(simulated[["date", "cout_1"]], on="date").rename(columns={"cout_1": "sim_m3s"})

ev = flows[flows.date >= evaluation_start]
o, s = ev.flow_m3s.to_numpy(), ev.sim_m3s.to_numpy()
nse = 1 - ((s - o) ** 2).sum() / ((o - o.mean()) ** 2).sum()
r = np.corrcoef(o, s)[0, 1]
bias = s.mean() / o.mean()
spread = s.std() / o.std()
kge = 1 - np.sqrt((r - 1) ** 2 + (bias - 1) ** 2 + (spread - 1) ** 2)
print(f"{ev.date.min().date()} to {ev.date.max().date()}:")
print(f"  NSE {nse:.2f}   KGE {kge:.2f}   r {r:.2f}   mean sim/obs {bias:.2f}   spread sim/obs {spread:.2f}")

In [ ]:
fig, (ax_rain, ax_q) = plt.subplots(2, 1, figsize=(11, 6), sharex=True,
                                    gridspec_kw={"height_ratios": [1, 3]})
ax_rain.bar(met.date, met.rainfall_mm, width=1.0, color="tab:blue")
ax_rain.invert_yaxis()
ax_rain.set_ylabel("Rainfall\n(mm/day)")

ax_q.plot(flows.date, flows.flow_m3s, lw=0.8, color="black", label=f"NRFA {station} (observed)")
ax_q.plot(flows.date, flows.sim_m3s, lw=0.8, color="tab:orange", label="S-HYPE (simulated)")
ax_q.axvspan(flows.date.min(), pd.Timestamp(evaluation_start), color="grey", alpha=0.15, label="spin-up")
ax_q.set_yscale("log")
ax_q.set_ylim(0.05, flows[["flow_m3s", "sim_m3s"]].max().max() * 1.5)
ax_q.set_ylabel("Flow (m³ s⁻¹)")
ax_q.legend(loc="lower left")
plt.tight_layout()
plt.show()

**Reading the result.** With uniform, uncalibrated parameters the timing of the response is roughly right: the flows rise with the rain and recede afterwards. The **volume** is too low: the simulated flow is well below the observed. Most of this is the forcing. Compare the rainfall and the gauged runoff in Step 4: when the river carries away about as much water as the rainfall series supplies, before any evaporation, the rainfall must be too low for these hills. No model parameter can make up missing rain.

That is typical of a first run, and it is where real modelling starts: better forcing, then the land use and soils of the catchment, then calibration of the parameters against the record.

### Why the simulated flow sometimes just recedes

Look at the hydrograph: there are long stretches where the simulated flow ignores the rain and falls steadily, sometimes to almost nothing, while the river responds. It is tempting to blame snow, but it is not snow. The model turns precipitation into snow only below +2 °C (all of it below −1 °C), and most of these stretches are in spring and summer, at 5–14 °C. Three things in this simple set-up cause it:

1. **The start.** The model begins with every soil layer exactly at field capacity, and its rivers empty. In HYPE, soil only drains to the stream from water held **above** field capacity, so at first there is nothing to drain: on dry days the runoff is exactly zero, and the flow empties away towards zero. This is the spin-up, and it is why we only score the model from the second year.
2. **Summer soil deficits.** The uniform soil is 1.5 m deep and holds about 330 mm of water that plants can use. In summer, evaporation draws it below field capacity, and then every rainfall first has to refill that deficit before any water drains to the stream. In a test run, the model produced 4 mm of runoff from 65 mm of rain between 24 June and 6 July 2023, while the gauge recorded 21 mm.
3. **No groundwater.** Between storms, the only outflow is the soil draining its water above field capacity, at the rates `rrcs1` and `rrcs2`. Once the soil is back at field capacity, the outflow stops. This version of the model has no groundwater store, and a real Southern Uplands river is kept running between storms by groundwater in fractured rock and glacial deposits.

**A thicker soil can stand in for groundwater.** Evaporation only takes water from the top two layers, so a thick third layer that reaches down to the stream depth (`layer3_bottom_m` and `stream_depth_m` in the class table), with a slow drainage rate `rrcs2`, holds water through dry spells and releases it slowly: baseflow. In a test with the third layer extended to 3 m and `rrcs2` = 0.008 per day, the lowest flow rose from 0.13 to 0.65 m³ s⁻¹. The observed low was 0.15, so those values were too slow, but the mechanism works. It is a stand-in: Ettrick's real soils are thin.

**Calibration can tune all of this**: the soil depths, the drainage rates, the storage. Step 8 calibrates seven parameters, and that improves the fit a lot. But it is still **one class**: every point in the catchment, valley floor or hilltop, behaves the same, so the calibrated values are effective values for the whole catchment, with no spatial element. Step 9 tries adding one.

## Step 8: Calibrating with SPOTPY (a quick run)

The parameters so far are a guess. **Calibration** searches for the values that make the simulated flow match the gauge best. We use [SPOTPY](https://github.com/thouska/spotpy), the Statistical Parameter Optimization Tool for Python (Houska et al., 2015): a package for calibrating models and analysing their uncertainty. It runs our model many times with different parameter values, and keeps a record of every run.

**The algorithm** is SCE-UA, the Shuffled Complex Evolution method of Duan et al. (1992), written for calibrating conceptual rainfall–runoff models and still a standard for them. It starts from a set of random parameter sets, divides them into groups (*complexes*) that each evolve towards better values, and shuffles the groups so they share what they have found.

**The score** is the KGE (Step 7). SCE-UA minimises, so we give it 1 − KGE.

**What we calibrate**: seven parameters, each searched within a range:

| Parameter | Range | What it controls |
|---|---|---|
| `rain_multiplier` | 1.0–1.6 | scales the whole rainfall series. Not a HYPE parameter: it stands in for the rainfall missing from the forcing (Step 4), so that the soil parameters are not distorted to make up for it |
| `rrcs1`, `rrcs2` | 0.05–0.6, 0.002–0.1 day⁻¹ | how fast the top and bottom soil layers drain to the stream |
| `wcfc`, `wcep` | 0.08–0.35, 0.03–0.3 | how much water the soil holds for plants, and how much can drain |
| `cevp` | 0.05–0.35 mm °C⁻¹ day⁻¹ | potential evaporation |
| `rivvel` | 0.2–2.0 m s⁻¹ | river velocity |

**Calibration and validation.** Each run simulates the spin-up year and the **calibration year**, 2022–23, and is scored on 2022–23 only. The final year, 2023–24, is kept back: we then run the best parameters on it to see whether the improvement is real, or the parameters have just been fitted to one year.

### A quick run, and what it costs

A full SCE-UA calibration with SPOTPY's defaults uses 20 complexes: for seven parameters that is 300 runs before the first round of evolution, and a few thousand runs in all. We cut it down:

* `ngs=5`: five complexes (75 runs per round) instead of twenty;
* `repetitions=600`: a cap on the number of runs (SPOTPY checks it between rounds, so it can go a little over);
* `kstop=3`, `pcento=0.01`: or stop sooner, when three rounds in a row improve the score by less than 1%.

On a laptop, one run of the model, including writing its files, took 0.2 s, and this calibration made about 410 runs in 80 seconds. Colab's free machines are slower: expect roughly **3 to 5 minutes**. (A full calibration with the defaults would take about ten minutes on the laptop, and half an hour or more on Colab.)

The price is a less thorough search: with fewer complexes, SCE-UA is likelier to settle on a good parameter set that is not the best, and the parameters are less well determined. The last part of this step shows how loosely.

In [ ]:
!pip install -q spotpy

In [ ]:
import subprocess
import time

import spotpy

calib_start, calib_end = "2022-10-01", "2023-09-30"
validation_start = "2023-10-01"
n_calib_days = (pd.Timestamp(calib_end) - pd.Timestamp(start_date)).days + 1   # spin-up + calibration

# A copy of the model's parameter file that reads the calibration tables and
# stops after the calibration year
with open("ettrick_hype.param") as f:
    calib_param = f.read()
for old, new in [("write_fname: ettrick", "write_fname: calib"),
                 ("hype_general.csv", "calib_general.csv"),
                 ("hype_landuse.csv", "calib_landuse.csv"),
                 ("hype_soil.csv", "calib_soil.csv"),
                 ("ettrick_rainfall.csv", "calib_rainfall.csv"),
                 ("write_hype_subbasin_raster: true\n", "")]:
    calib_param = calib_param.replace(old, new)
calib_param += f"n_timesteps: {n_calib_days}\n"
with open("calib.param", "w") as f:
    f.write(calib_param)

base_general = pd.read_csv("hype_general.csv").set_index("name")["value"]
base_landuse = pd.read_csv("hype_landuse.csv")
base_soil = pd.read_csv("hype_soil.csv")
base_rain = pd.read_csv("ettrick_rainfall.csv")


def write_tables(p, tag):
    """Write the rainfall and parameter tables for the parameter values in dict p."""
    rain = base_rain.copy()
    rain["rainfall_mm"] = rain["rainfall_mm"] * p.get("rain_multiplier", 1.0)
    rain.to_csv(f"{tag}_rainfall.csv", index=False)
    general = base_general.copy()
    general["rivvel"] = p.get("rivvel", general["rivvel"])
    general.rename_axis("name").reset_index().to_csv(f"{tag}_general.csv", index=False)
    landuse = base_landuse.copy()
    landuse["cevp"] = p.get("cevp", landuse["cevp"].iloc[0])
    landuse.to_csv(f"{tag}_landuse.csv", index=False)
    soil = base_soil.copy()
    for key in ("rrcs1", "rrcs2", "wcfc", "wcep"):
        soil[key] = p.get(key, soil[key].iloc[0])
    soil.to_csv(f"{tag}_soil.csv", index=False)


calib_observed = observed[(observed.date >= calib_start) & (observed.date <= calib_end)]


class EttrickSetup:
    """The model as SPOTPY sees it: parameters, a simulation, observations and a score."""

    def __init__(self):
        self.params = [
            spotpy.parameter.Uniform("rain_multiplier", 1.0, 1.6),
            spotpy.parameter.Uniform("rrcs1", 0.05, 0.6),
            spotpy.parameter.Uniform("rrcs2", 0.002, 0.1),
            spotpy.parameter.Uniform("wcfc", 0.08, 0.35),
            spotpy.parameter.Uniform("wcep", 0.03, 0.3),
            spotpy.parameter.Uniform("cevp", 0.05, 0.35),
            spotpy.parameter.Uniform("rivvel", 0.2, 2.0),
        ]
        self.names = [p.name for p in self.params]

    def parameters(self):
        return spotpy.parameter.generate(self.params)

    def simulation(self, x):
        write_tables(dict(zip(self.names, x)), "calib")
        run = subprocess.run(["lsdtt-distributed-hydrology", "./", "calib.param"],
                             capture_output=True, text=True)
        if run.returncode != 0:                       # a failed run gets the worst score
            return [np.nan] * len(calib_observed)
        sim = pd.read_csv("calib_hype_discharge.csv", parse_dates=["date"])
        sim = sim[(sim.date >= calib_start) & (sim.date <= calib_end)]
        return sim["cout_1"].tolist()

    def evaluation(self):
        return calib_observed["flow_m3s"].tolist()

    def objectivefunction(self, simulation, evaluation, params=None):
        if np.any(np.isnan(simulation)):
            return 10.0
        return 1.0 - spotpy.objectivefunctions.kge(evaluation, simulation)


started = time.time()
sampler = spotpy.algorithms.sceua(EttrickSetup(), dbname="calib_runs", dbformat="csv", save_sim=False)
sampler.sample(600, ngs=5, kstop=3, pcento=0.01, peps=0.01)
runs = spotpy.analyser.load_csv_results("calib_runs")
print(f"\nCalibration took {time.time() - started:.0f} s; {len(runs)} runs saved")

### How the search went

Each dot is one run that SPOTPY saved (it does not save the trial runs inside each round of evolution); the line is the best score found so far. The search improves quickly at first, then levels off, which is when SCE-UA stops.

In [ ]:
score = 1 - runs["like1"]                       # back from 1 - KGE to KGE
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(score, ".", color="grey", alpha=0.6, label="each run")
ax.plot(np.maximum.accumulate(score), color="tab:orange", lw=2, label="best so far")
ax.set_xlabel("Run")
ax.set_ylabel("KGE, calibration year")
ax.set_ylim(max(score.min(), -1), 1)
ax.legend()
plt.show()

best_row = runs[np.argmin(runs["like1"])]
best = {name[3:]: float(best_row[name]) for name in runs.dtype.names if name.startswith("par")}
pd.Series(best).round(3).rename("best value").to_frame()

### Calibration against validation

Now run the full three years twice, with the starting parameters and with the best calibrated ones, and score both on the calibration year and on the validation year that the calibration never saw.

In [ ]:
def run_full(p, tag):
    """Run all three years with parameter values p; return the simulated flow."""
    write_tables(p, tag)
    with open("ettrick_hype.param") as f:
        text = f.read()
    for old, new in [("write_fname: ettrick", f"write_fname: {tag}"),
                     ("hype_general.csv", f"{tag}_general.csv"),
                     ("hype_landuse.csv", f"{tag}_landuse.csv"),
                     ("hype_soil.csv", f"{tag}_soil.csv"),
                     ("ettrick_rainfall.csv", f"{tag}_rainfall.csv")]:
        text = text.replace(old, new)
    with open(f"{tag}.param", "w") as f:
        f.write(text)
    subprocess.run(["lsdtt-distributed-hydrology", "./", f"{tag}.param"], capture_output=True, check=True)
    sim = pd.read_csv(f"{tag}_hype_discharge.csv", parse_dates=["date"])
    return sim[["date", "cout_1"]].rename(columns={"cout_1": tag})


def scores(sim_col, start, end):
    period = comparison[(comparison.date >= start) & (comparison.date <= end)]
    o, s = period.flow_m3s.to_numpy(), period[sim_col].to_numpy()
    return {"KGE": spotpy.objectivefunctions.kge(o, s),
            "NSE": 1 - ((s - o) ** 2).sum() / ((o - o.mean()) ** 2).sum(),
            "mean sim/obs": s.mean() / o.mean()}


comparison = (observed.merge(run_full({}, "uncalibrated"), on="date")
                      .merge(run_full(best, "calibrated"), on="date"))
table = {}
for col in ("uncalibrated", "calibrated"):
    table[(col, "calibration 2022-23")] = scores(col, calib_start, calib_end)
    table[(col, "validation 2023-24")] = scores(col, validation_start, end_date)
pd.DataFrame(table).T.round(2)

In [ ]:
v = comparison[comparison.date >= validation_start]
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(v.date, v.flow_m3s, color="black", lw=0.9, label="observed")
ax.plot(v.date, v.uncalibrated, color="tab:blue", lw=0.8, label="uncalibrated")
ax.plot(v.date, v.calibrated, color="tab:orange", lw=0.9, label="calibrated")
ax.set_yscale("log")
ax.set_ylim(0.05, v[["flow_m3s", "calibrated"]].max().max() * 1.5)
ax.set_ylabel("Flow (m³ s⁻¹)")
ax.set_title("Validation year, 2023-24: not used in the calibration")
ax.legend(loc="lower left")
plt.show()

### How certain are the parameters?

Many quite different parameter sets fit almost as well as the best one: hydrologists call this **equifinality**, and it is normal for a model like HYPE calibrated on one gauge. Below, each parameter's range is scaled from 0 (bottom of its search range) to 1 (top), and the dots are the runs whose KGE is within 0.05 of the best. A parameter whose dots spread across most of its range is **poorly determined**: the gauge does not tell us its value.

A quick run makes this worse: with only a few hundred runs, the near-best sets are few and the picture is rough. (Only the runs SPOTPY saved are shown.) For a firmer answer, run SCE-UA with its defaults (`ngs=20`, `repetitions` of a few thousand) and leave it for half an hour or more, or use one of SPOTPY's Bayesian samplers (DREAM) to estimate the parameter uncertainty directly.

In [ ]:
setup = EttrickSetup()
near_best = runs[runs["like1"] <= runs["like1"].min() + 0.05]
print(f"{len(near_best)} of {len(runs)} runs have a KGE within 0.05 of the best")

fig, ax = plt.subplots(figsize=(8, 4))
for i, prm in enumerate(setup.params):
    values = (near_best["par" + prm.name] - prm.minbound) / (prm.maxbound - prm.minbound)
    ax.plot(values, np.full(len(values), i), "o", color="tab:orange", alpha=0.7)
    ax.plot((best[prm.name] - prm.minbound) / (prm.maxbound - prm.minbound), i, "k|", ms=18, mew=2)
ax.set_yticks(range(len(setup.params)))
ax.set_yticklabels([prm.name for prm in setup.params])
ax.set_xlim(-0.02, 1.02)
ax.set_xlabel("Position in the search range (0 = low end, 1 = high end); black bar: best")
plt.tight_layout()
plt.show()

**Reading the result.** The calibrated model should do much better than the starting parameters on the calibration year, and still clearly better on the validation year, which it never saw. (In a test run on a laptop, the KGE went from 0.29 to 0.82 on the calibration year and from 0.17 to 0.76 on the validation year. Your numbers will differ a little, because the search starts from random parameter sets.) Expect the rainfall multiplier to settle well above 1: the calibration confirms that the forcing is short of rain. And expect several parameters to stay poorly determined: one gauge and one year cannot pin down seven parameters.

## Step 9: A second soil class along the channels

In a real catchment the ground next to the streams behaves differently from the hillsides: valley floors, flushes and peat stay wet all year, and in summer they produce storm runoff while the hillsides soak up the rain. A natural way to give the model a spatial element is a **second soil class**: a thin, wet soil in a **buffer** along the channel network, and the original soil everywhere else.

This is also how land-use and soil maps go into the model in general: as integer codes on the DEM grid. We build the soil raster from the channel lines of Step 3:

1. **buffer** the channel lines by 60 m on each side (`geopandas`);
2. **rasterise** the buffer onto the DEM grid (`rasterio.features.rasterize`): soil code 2 inside it, 1 outside;
3. give soil 2 its own row in the soil table, and the new (land use 1, soil 2) pair a row in the class table.

The near-stream soil is thin (two layers, 0.1 and 0.3 m, stream depth 0.3 m), holds little water for plants (`wcfc` = 0.10), and drains fast (`rrcs1` = 0.6, `rrcs2` = 0.3 per day). Everything else is the uncalibrated set-up of Step 6, so the only difference is the second class.

In [ ]:
from rasterio import features

buffer_m = 60
near_stream = features.rasterize(
    [(geom, 1) for geom in network.buffer(buffer_m).geometry],
    out_shape=dem.shape, transform=rasterio.open("ettrick_DEM.tif").transform, fill=0).astype(bool)
in_catchment = features.rasterize(
    [(geom, 1) for geom in catchment.geometry],
    out_shape=dem.shape, transform=rasterio.open("ettrick_DEM.tif").transform, fill=0).astype(bool)
print(f"The near-stream class covers {100 * (near_stream & in_catchment).sum() / in_catchment.sum():.1f}% of the catchment")

with rasterio.open("ettrick_soil_buffer.tif", "w", **profile) as dst:
    dst.write(np.where(valid, np.where(near_stream, 2, 1), -9999).astype("int32"), 1)

near_stream_soil = {"code": 2, "wcwp": 0.08, "wcfc": 0.10, "wcep": 0.20,
                    "rrcs1": 0.6, "rrcs2": 0.3, "mperc1": 30, "mperc2": 15,
                    "macrate": 0.1, "srrate": 0.2, "mactrinf": 5, "mactrsm": 0.5}
pd.concat([pd.read_csv("hype_soil.csv"), pd.DataFrame([near_stream_soil])]).to_csv("buffer_soil.csv", index=False)
pd.concat([pd.read_csv("hype_classes.csv"),
           pd.DataFrame([{"class_id": 2, "landuse": 1, "soil": 2, "n_layers": 2,
                          "layer1_bottom_m": 0.1, "layer2_bottom_m": 0.3, "stream_depth_m": 0.3}])]
          ).to_csv("buffer_classes.csv", index=False)
print(open("buffer_classes.csv").read())

with open("ettrick_hype.param") as f:
    text = f.read()
for old, new in [("write_fname: ettrick", "write_fname: buffer"),
                 ("ettrick_soil.tif", "ettrick_soil_buffer.tif"),
                 ("hype_soil.csv", "buffer_soil.csv"),
                 ("hype_classes.csv", "buffer_classes.csv")]:
    text = text.replace(old, new)
with open("buffer.param", "w") as f:
    f.write(text)

!lsdtt-distributed-hydrology ./ buffer.param

In [ ]:
two_classes = pd.read_csv("buffer_hype_discharge.csv", parse_dates=["date"])[["date", "cout_1"]]
both = flows.merge(two_classes.rename(columns={"cout_1": "buffer_m3s"}), on="date")

ev2 = both[both.date >= evaluation_start]
for col, label in (("sim_m3s", "one class"), ("buffer_m3s", "two classes")):
    o, s = ev2.flow_m3s.to_numpy(), ev2[col].to_numpy()
    print(f"{label:12s} KGE {spotpy.objectivefunctions.kge(o, s):.2f}   lowest flow {s.min():.2f} m3/s")

storm = both[(both.date >= "2023-06-15") & (both.date <= "2023-07-15")]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(storm.date, storm.flow_m3s, color="black", label="observed")
ax.plot(storm.date, storm.sim_m3s, color="tab:blue", label="one class")
ax.plot(storm.date, storm.buffer_m3s, color="tab:green", label="two classes (near-stream buffer)")
ax.set_ylabel("Flow (m³ s⁻¹)")
ax.set_title("Summer 2023")
ax.legend()
plt.show()

**It barely helps.** In a test run, the 60 m buffer (about a tenth of the catchment) lifted the summer storm peak of late June 2023 from 0.18 to 0.26 m³ s⁻¹, against 2.43 observed, and KGE from 0.23 to 0.25. Even a near-stream soil that can hold almost no water, and sends most heavy rain over the surface, only reached 0.54 m³ s⁻¹.

The reason is in how HYPE is built. **Classes do not exchange water**: each one only receives the rain that falls on it, and the near-stream class is a tenth of the catchment. All of the wettest day's rain in that spell (10.4 mm) running off a tenth of the catchment at once would be a daily mean of only 0.5 m³ s⁻¹. In the real catchment the valley floors stay wet because water **drains into them from the hillslopes** above, so in summer the rain falling on them, and the water pushed out of them, reaches the river quickly. A HYPE class cannot be fed by the classes upslope of it.

So a spatial element needs more than another class. In HYPE itself, the usual answer is calibration of effective parameters, with features this version leaves out (regional groundwater, for example). A model built around water moving downslope to wet valley floors is TOPMODEL, the other `column_model` of `lsdtt-distributed-hydrology`, which predicts where the catchment saturates from its topography. (Part of the gap may also be the forcing: a single reanalysis point can miss local summer storms.)

## Things to try

* **Calibrate for longer.** In Step 8, raise `ngs` to 10 and `repetitions` to 2000. Does the best KGE improve, and do the parameters become better determined?
* **Calibrate on a different score.** NSE weights the floods most; KGE on the logarithm of the flow weights the low flows. Change `objectivefunction` and compare the parameter sets.
* **Change the soil.** A deeper or more porous soil (`wcfc`, `wcep`, the layer depths in `hype_classes.csv`) stores more water and smooths the response; faster `rrcs1` makes it flashier.
* **Change the river.** `rivvel` and `damp` control how quickly the runoff reaches the gauge.
* **Snow.** Winter 2022–23 had cold spells: look at the temperature series and at the flows after them. `cmlt` and `ttmp` control melt.
* **Sub-basins.** Add more points to `ettrick_gauge.csv` (latitude, longitude, point_id), for example at tributary confluences: each becomes a sub-basin with its own outflow column in `ettrick_hype_discharge.csv`.
* **Another catchment.** Change `station` in Step 1 to another NRFA gauge (find one on the [NRFA data explorer](https://nrfa.ceh.ac.uk/data/explorer)), and move the box in Step 2 so it contains the whole catchment.
* **Hand it to HYPE.** Add `write_hype_geodata: true` to the parameter file: the set-up is written as HYPE's `GeoData.txt` and `GeoClass.txt`, so the same catchment can be run in SMHI's HYPE.

## Data and references

* Daily flow and station data: UK National River Flow Archive, station 21017 (https://nrfa.ceh.ac.uk/data/station/info/21017). Data © UKCEH, used under the NRFA terms.
* Weather: Open-Meteo historical weather API (https://open-meteo.com), based on ERA5 and other reanalysis and model data (CC BY 4.0).
* DEM: Copernicus GLO-30 (© DLR e.V. 2010–2014 and © Airbus Defence and Space GmbH 2014–2018, provided under COPERNICUS by the European Union and ESA).
* Lindström, G., Pers, C., Rosberg, J., Strömqvist, J. and Arheimer, B. (2010). Development and testing of the HYPE (Hydrological Predictions for the Environment) water quality model for different spatial scales. *Hydrology Research*, 41(3–4), 295–319. https://doi.org/10.2166/nh.2010.007
* Strömqvist, J., Arheimer, B., Dahné, J., Donnelly, C. and Lindström, G. (2012). Water and nutrient predictions in ungauged basins: set-up and evaluation of a model at the national scale. *Hydrological Sciences Journal*, 57(2), 229–247. https://doi.org/10.1080/02626667.2011.637497
* Houska, T., Kraft, P., Chamorro-Chavez, A. and Breuer, L. (2015). SPOTting model parameters using a ready-made Python package. *PLOS ONE*, 10(12), e0145180. https://doi.org/10.1371/journal.pone.0145180
* Duan, Q., Sorooshian, S. and Gupta, V. (1992). Effective and efficient global optimization for conceptual rainfall-runoff models. *Water Resources Research*, 28(4), 1015–1031. https://doi.org/10.1029/91WR02985
* Nash, J. E. and Sutcliffe, J. V. (1970). River flow forecasting through conceptual models part I — A discussion of principles. *Journal of Hydrology*, 10(3), 282–290. https://doi.org/10.1016/0022-1694(70)90255-6
* Gupta, H. V., Kling, H., Yilmaz, K. K. and Martinez, G. F. (2009). Decomposition of the mean squared error and NSE performance criteria: Implications for improving hydrological modelling. *Journal of Hydrology*, 377(1–2), 80–91. https://doi.org/10.1016/j.jhydrol.2009.08.003